<a href="https://colab.research.google.com/github/ssykes-eth/ETH_275-0005-00L/blob/code_exercises/05_dgd_exercise_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One Model, Two Thousand Machines

**The setting.** A training run has been approved and somebody has to say whether it can be
scheduled. You have the model, the data, the deadline and the machines. What you do not have yet is
an answer, and the meeting is on Monday.

**How we will work.** Nothing here is measured, because measuring means booking two thousand
machines for a week. Everything is worked out from the numbers you were given, which this notebook
calls **the brief**. Every time in it is what the run would take if nothing went wrong, so treat
them as a plan rather than a measurement.

**What you will end up with.** Six questions you can ask about any proposed configuration, the
arithmetic behind each one, and a defensible answer for Monday.

In [ ]:
#@title 📥 0.1 — fetch the files (run me first) { display-mode: "form" }
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE7-public"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    token = ""
    try:                                  # private repo (testing): read token from Secrets
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:                     # public repo (students): no token needed
        token = ""
    auth = f"{token}@" if token else ""
    url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo...")
        !git clone -q "$url"
    else:                                 # already cloned: refresh it to the latest version
        print("Updating the exercise repo to the latest version...")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull, so using the existing copy)"

# Find the folder holding the helper files. Searching for it rather than naming a
# path means the repo can be laid out however it likes without breaking this cell.
def _find_src(*roots):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for base, _dirs, files in os.walk(root):
            if os.path.basename(base) == "src" and "runviz.py" in files:
                return os.path.abspath(base)
    return None

SRC = _find_src(REPO_NAME, ".", "..", "../..")
if SRC is None:
    # Last resort (public repo, git unavailable): fetch just the files we need.
    try:
        from urllib.request import urlretrieve
        SRC = os.path.abspath("src")
        os.makedirs(SRC, exist_ok=True)
        base = (f"https://raw.githubusercontent.com/{REPO_OWNER}/{REPO_NAME}/main/"
                "5_dgd/src/")
        for fname in ("brief.py", "cluster_sim.py", "quizzes.py", "runcore.py", "runviz.py"):
            urlretrieve(base + fname, os.path.join(SRC, fname))
        print("Fetched the exercise files directly.")
    except Exception:
        raise FileNotFoundError(
            "Could not find the exercise files. If the repo is still private, add a "
            "GITHUB_TOKEN secret (see the note above) and re-run this cell.")

os.chdir(os.path.dirname(SRC))
sys.path.insert(0, SRC)
print("Working directory:", os.getcwd())

In [ ]:
#@title 📦 0.2 — set up { display-mode: "form" }
%config InlineBackend.figure_format = "svg"
import numpy as np
import brief as the_brief
import runcore as rc
import runviz as rv
import cluster_sim as sim
print("set up")

In [ ]:
#@title 📋 0.3 — the run you have been asked to schedule { display-mode: "form" }
rv.the_brief()

### A parameter, a token, and how much text two trillion is

A **parameter** is one of the numbers inside the model. Training does not change what the model is
made of, it changes those numbers, one small nudge at a time. There are seventy billion of them.

A **token** is one piece of the text the model reads, usually a common word or a fragment of a
longer one. In English it works out at roughly three quarters of a word each. So two trillion
tokens is about one and a half trillion words, and it is worth pausing on what that is: a
three hundred page book holds around a hundred thousand words, so **the run reads the equivalent of
about fifteen million books**. A person reading eight hours a day would need something like thirty
thousand years to get through it.

Of everything in the brief, the twenty five days and the three quarters are the only two that are
wishes rather than facts. Somebody asked for them, and part of your job is knowing what they cost.

### How the machines are wired together

A **machine** here is one accelerator. They come in **boxes** of eight, so 2,048 machines is 256
boxes. Inside a box the machines talk to each other at 900 GB/s. Between boxes, across what is
called the **fabric**, it is 25 GB/s.

That gap is thirty six times, and it decides more in the second half of this notebook than any
other number.

### Six questions, and you will write the arithmetic behind every one

The notebook ends on a panel that scores any proposed configuration against six gates, and every
one of them runs on a function you wrote on the way there.

**How to work through it.** Run the cells in order. Eight of them are marked 🎯 **Task 1** through
**Task 8**: each has `???` where a line is missing, a hint just above it, and a check that says in
words whether it worked. They are in the outline, so you can always find your place again.

**Where this goes.** Seven parts, each one question.

| | The part asks | What we look at |
|---|---|---|
| 1 | Can one machine do it? | The machines you were given, how much arithmetic a run is, and the two rates that turn it into time |
| 2 | Does splitting the work change the answer? | Whether four machines give the same answer as one, and when they quietly do not |
| 3 | What does it cost the machines to agree? | What they exchange every step, what it costs, and the code that does it |
| 4 | Does the model even fit? | What a machine must hold for every parameter in order to improve it |
| 5 | Three ways to cut a model that does not fit | Across a layer, along the stack, and the copies nobody is reading |
| 6 | Which configuration do we run? | Six gates, two dials, and the gap between valid and worth starting |
| 7 | What does the plan not know? | A slow machine, a run that stops, and a requirement that changes |

Before you can schedule a run you need to know how much work is in it, so that is where we start.

---
# Part 1 · Can one machine do it?

Start with the smallest possible plan. One machine, no coordination, no network, nothing to go
wrong. If that works, everything after this is unnecessary.

The brief says 2,048 machines of 96 GB, in boxes of eight. That is a line on a page until you look,
and looking is the first thing anybody does.

The box below answers as one of those 256 would, and it calls its machines GPUs because that is
what the tools call them.

### 🎯 Task 1 · find out what you have been given

> 💡 **Hint** · The command is `nvidia-smi`, and it goes in as a string, quotes and all:
> `sim.console("nvidia-smi")`. It is the first thing anybody types on a machine they have just been
> given, and it reports the model, the memory and what is running on each one.

In [ ]:
# 🎯 TASK 1 — find out what you have been given
sim.console(???)          # what has this box actually got in it?

Eight machines of 96 GB, idle, exactly as the brief said.

> ⚠️ **We built that box for you, and it is not real.** There is no hardware behind that table and
> nothing was queried. `cluster_sim.py` is a short file written for this exercise: it prints output
> in the shape `nvidia-smi` really prints it, filled in with the numbers from the brief. The command
> is real and worth knowing. The machine answering it is not, and nothing in this notebook needs a
> GPU to run.

So: what can one of those machines do?

A training run is a fixed quantity of arithmetic, and you can count it before naming any hardware.
The counting starts with what happens to one token.

```
for every token:

  going forward    use the model to predict what comes next.
                   every parameter is multiplied by something,
                   and the result is added on                            2 operations

  coming back      for every parameter, two separate questions
                   get answered, and each is another multiply and add:

                   what should this parameter become?                    2 operations
                   what should the layer before it be told?              2 operations
                                                                        ─────────────
                   for every parameter, for every token                  6 operations
```

Six is not exact and does not need to be. What matters is that it does not depend on the cluster,
the schedule, or anything you get to decide. Multiply it by the parameters and the tokens and you
have the whole job, before a single machine has been named.

In [ ]:
#@title 🧮 1.1 — the size of the job { display-mode: "form" }
rv.size_of_the_job()

### Seventy six years, and only one number in it came from the hardware

Three of the four terms are settled before any hardware is chosen: six operations a parameter a
token is how training works, and seventy billion and two trillion are the brief. Only the last
step, turning operations into seconds, asks anything of the machine, and it asks twice.

The two rates are worth naming separately, because people use them interchangeably and they are
not the same thing at all.

### The peak rate, the sustained share, and why only one of them is knowable

The **peak rate** is what the machine is sold with: the arithmetic it can do in a second when
everything lines up perfectly. The brief says a thousand trillion operations a second.

The **sustained share** is the fraction of that a real run actually achieves. Data has to be
fetched before it can be used and results have to be written back, and while either is happening
the arithmetic units have nothing to do. The brief says thirty five percent.

Those two multiplied together are the only hardware in the calculation:

$$\text{seconds} = \frac{\text{operations in the whole run}}
{\text{peak rate} \times \text{sustained share}}$$

In words: take the job, and divide it by what the machine really manages per second rather than
what it says on the box. The bar in the figure above is that division drawn out. Twenty seven of
the seventy six years is arithmetic running at the advertised rate. The other forty nine is the
machine waiting on itself.

### Nobody can look up the sustained share

Move that fraction to one and the run takes twenty seven years, which is what the datasheet implies
and what no real run has reached. Move it to a fifth and it takes a hundred and thirty three.

That is a large amount of answer resting on one fraction, and the fraction is not a property of the
machine. It depends on how the model is shaped, how the data arrives and how the work is ordered,
so two runs on identical hardware do not get the same number. The only way to know it for your run
is to run a small piece and measure.

Which is worth remembering when somebody offers you a faster machine.

In [ ]:
#@title 🧠 Quick check — a chip with double the peak rate { display-mode: "form" }
rv.quick_check("faster_chip")

One machine is out, by a factor of a thousand, so the plan needs more of them. Getting more of them
working is one command, and the command takes one number: how many processes to start on this box.

That number is not free to choose. A process takes a machine for itself, so eight machines want
eight processes. Ask for four and half the box sits idle. Ask for sixteen and eight of them have
nowhere to run.

### 🎯 Task 2 · start one process for each machine

> 💡 **Hint** · It is `8`, one process for each machine `nvidia-smi` just listed. Put a wrong number
> in first if you like, and the box will tell you what it did with it.

In [ ]:
# 🎯 TASK 2 — start one process for each machine
processes = ???          # one for every machine in the box
sim.console(f"torchrun --nproc-per-node={processes} train.py")

Eight processes, one to a machine, and each one told three things: which machine it is on
(**`LOCAL_RANK`**), its number across the whole run (**`RANK`**) and how many processes there are
altogether (**`WORLD_SIZE`**). Those three numbers are the entire interface between the launcher and
your code, and every line of the training script in Part 3 reads one of them.

> ⚠️ **Nothing launched.** Same file, same pretence. `torchrun` is the real command and
> `--nproc-per-node` is the real flag, but the reply was printed, not earned. What is worth carrying
> out of these two cells is the shape of the commands and what each one tells you.

So the work is on eight machines instead of one, which is the only way this run ever finishes.
Whether spreading it out changes the answer is the next question.

---
# Part 2 · Does splitting the work change the answer?

Before spreading a job over two thousand machines it is worth knowing whether the answer survives
the journey. This part uses eight numbers instead of seventy billion, small enough that you can
check every claim by eye.

Think of each number as one example's opinion about which way the model should move. Eight
examples, eight opinions, and training uses their average.

In [ ]:
#@title 📊 2.1 — the same eight, split two ways { display-mode: "form" }
rv.splitting_the_batch()

### Machine 1 sees zero

Start with the left panel, where the split is even and everything works. Machine 1's two examples
pointed in opposite directions and cancelled, so the direction it computed is zero. Nothing it can
see suggests there is any more work to do, and it has no way of finding out otherwise. The answer
over all eight examples is 3.0, and **no machine computed it**.
It exists only in the average of the four, which is to say only after they talk to each other.

This is why the combining step is not an optimisation. A machine that skips it is not being
slightly less accurate. It is taking a step in a direction that nobody checked, on evidence it
cannot know is partial.

### A reasonable looking split, wrong by seven percent

Now the right panel. Nothing about a three, two, one and two split is strange: a scheduler giving a
fast machine a little more work produces exactly it. The examples are the same eight and the
machines are the same four, but machine 1 now holds a third one, machine 3 holds a single example,
and a plain average of the four answers comes out at 3.2083 instead of 3.0. Seven percent is not a rounding error. It is the cluster steadily
optimising something slightly different from what it was asked to, in a way no output would show
you.

### The weights are what the arithmetic requires, not a policy

The four machines held two examples each, which is why a plain average of their four answers came
out right. That is a coincidence of the split, and it is worth seeing what the general rule is.

Training uses the average over every example:

$$g = \frac{1}{n}\sum_{i=1}^{n} g_i$$

Here $n$ is how many examples the batch holds, $g_i$ is the direction from example $i$ on its own,
and $g$ is the direction the whole batch would take: add up what every example says, divide by how
many there are.

Now group them by the machine holding them. Machine $k$ holds $n_k$ examples, and the average over
just those is $g_k$. Splitting the sum by machine gives:

$$g = \sum_k \frac{n_k}{n}\, g_k$$

In words: the whole-batch direction is the average of the machines' directions, each counting in
proportion to how many examples it holds. Equal shares make those fractions equal and it reduces to
a plain average. Unequal shares do not.

### 🎯 Task 3 · combine four machines into one answer

> 💡 **Hint** · Line one: each machine's count divided by the total count, so the weights add up to
> one. Line two: multiply every machine's row by its own weight, then add the rows together. That is
> the equation above, and the only work left is saying it in code.

In [ ]:

def combine(directions, sizes):
    """Turn one direction per machine into the direction the whole batch would have given."""
    directions = np.asarray(directions, float)      # one row per machine
    sizes      = np.asarray(sizes, float)
    # 🎯 TASK 3 — combine the machines' directions into the whole-batch answer
    weights    = ???                 # each machine's share of all the examples
    return ???                       # the rows, added up with those weights


_even   = combine([[0., 1., -2.], [2., 1., 0.], [4., 3., 2.], [6., 3., 4.]], [2, 2, 2, 2])
_ragged = combine([[1/3, 1., -1.], [4.5, 2., 1.], [2., 3., 5.], [6., 3., 2.]], [3, 2, 1, 2])
print("four equal slices  ", np.round(_even, 4))
print("a ragged split     ", np.round(_ragged, 4))
print("both match the whole batch ✓"
      if np.allclose(_even, [3., 2., 1.]) and np.allclose(_ragged, [3., 2., 1.])
      else "✗ one of them is off: the weights are what make a ragged split come out right")

In [ ]:
#@title 🔬 2.2 — the same check on a real gradient { display-mode: "form" }
_dirs, _counts, _whole = rc.toy_gradients()
if not callable(globals().get("combine")):
    print("Not yet: combine. Write it in the cell above, run it, then run this one again.")
else:
    _combined = combine(_dirs, _counts)
    print(f"largest disagreement with the whole batch: {np.abs(_combined - _whole).max():.2e}")

In [ ]:
#@title 🧠 Quick check — a machine is given a bigger slice { display-mode: "form" }
rv.quick_check("bigger_slice")

So splitting the work is free, in the sense that it costs nothing in accuracy. Every machine still has
to end the step holding the same combined answer, and that is not free at all.

---
# Part 3 · What does it cost the machines to agree?

Part 2 used four machines and eight numbers. Two thousand machines and seventy billion parameters
work the same way, and this part is what that costs.

Everything here rests on one thing. **In this design every machine keeps a full copy of the
model.** They differ only in which examples they see, which is why they have to agree at the end of
every step: copies that took different steps stop being copies.

What they exchange is what Part 2 called a direction. At this size a direction is one number for
every parameter, so seventy billion numbers, and that block of numbers is called a **buffer**. Held
at two bytes each it comes to 140 GB.

This notebook calls that exchange **agreeing**, since that is what it is for. Elsewhere you will
hear it called communication, and this particular pattern an **all-reduce**.

So: 140 GB, reconciled across the whole cluster, every step. There are two obvious ways to arrange
that, and only one of them survives the cluster getting bigger.

In [ ]:
#@title 📊 3.1 — two ways to wire the same four machines { display-mode: "form" }
rv.hub_and_ring()

### The honest measure is what the busiest machine carries

It is tempting to count messages, and it tells you nothing. What decides whether a design survives
is the volume through whichever machine has the most to do, because everyone else waits for it.

A hub fails that test in the worst possible way: its own load grows with every machine you add. The
design gets worse exactly when you need it to get better.

A ring gives nobody that job, and it pays for that with two passes instead of one. The first pass
**adds**: a machine hands a slice on and whoever receives it adds their own, so one lap short of a
circle later each machine owns a slice that is finished. The second pass only **copies** those
finished slices round again.

What you are about to work out is what those two passes cost.

### 🎯 Task 4 · price one round of agreeing

> 💡 **Hint** · Line one: two passes, and each pass takes a hand-off for every machine except the
> one it starts at. Line two: the buffer split evenly, one slice for each machine. Neither line needs
> the other, and neither one is the buffer itself.

In [ ]:

def bytes_sent_per_machine(machines, buffer_gb):
    """How much ONE machine transmits so every machine ends up holding the same total.

    Transmitted only. The same amount arrives at it, and we count the two separately.
    """
    # 🎯 TASK 4 — what one machine sends in a round of agreeing
    #   hand_offs: two passes, and each pass is (machines - 1) hand-offs
    #   slice_gb:  buffer_gb shared out, one slice for each machine
    hand_offs = ???              # two passes round the group, each one short of a full lap
    slice_gb  = ???              # the buffer, split evenly across the machines
    return hand_offs * slice_gb


for _m in (2, 8, 64, 256, 1024, 2048):
    print(f"{_m:>5} machines   {bytes_sent_per_machine(_m, 140):>7.1f} GB transmitted   "
          f"{2 * (_m - 1):>5} hand-offs")
_ok = (abs(bytes_sent_per_machine(2, 140) - 140) < 1e-9
       and abs(bytes_sent_per_machine(8, 140) - 245) < 1e-9)
print("\n140 GB at two machines, 245 at eight, and never past 280 ✓" if _ok
      else "\n✗ two machines should transmit 140 GB and eight should transmit 245. "
           "One pass round the group gets you half of each")

### The volume stops growing and the waiting does not

Two things happened in that table and they point in opposite directions.

The volume each machine transmits **flattens out**. Going from two machines to two thousand takes it
from 140 GB to 280, and it never gets past twice the buffer however many you add. Each machine
sends more slices as the cluster grows, and each slice is smaller, and the two almost cancel.

The number of hand-offs **grows in a straight line**, and the reason is in the two passes: each one
takes a hand-off for every machine but the first, so it is two lots of one fewer than the machines.
At 2,048 that is 4,094 hand-offs, and every one is a wait before anything arrives. Our model
charges for the bytes and not for the waits, so every number in this notebook flatters.

### Two clocks, and only one of them speeds up

A step is two things happening one after the other:

$$\text{step} = \frac{\text{arithmetic}}{\text{machines}} + \text{agreeing}$$

The arithmetic divides cleanly, because every machine takes a share. The agreeing is what you just
computed, and it barely moves.

In [ ]:
#@title 🧮 3.2 — the two clocks, priced { display-mode: "form" }
def step_seconds(machines, work_flop, rate_flops, buffer_gb, link_gbs):
    """Modelled seconds for one training step: the arithmetic, plus the agreeing."""
    arithmetic = work_flop / rate_flops / machines   # the whole step, shared out
    agreeing   = bytes_sent_per_machine(machines, buffer_gb) / link_gbs
    return arithmetic + agreeing


STEP_FLOP, RATE = rc.step_flop(), the_brief.RATE_FLOPS
if not callable(globals().get("bytes_sent_per_machine")):
    print("Not yet: bytes_sent_per_machine. Write it above, run it, then run this one again.")
else:
    _one  = step_seconds(1,    STEP_FLOP, RATE, 140, 25)
    _many = step_seconds(2048, STEP_FLOP, RATE, 140, 25)
    print(f"one machine     {_one:>9,.1f} s a step")
    print(f"2,048 machines  {_many:>9,.1f} s a step, "
          f"of which {_many - STEP_FLOP/RATE/2048:.1f} is agreeing")
    print(f"\nthe arithmetic divided by {the_brief.MACHINES:,}. The agreeing did not move.")

In [ ]:
#@title 📈 3.3 — where adding machines stops paying { display-mode: "form" }
rv.efficiency_curve()

### We own two thousand machines and this way of working can use three hundred

At the brief's target the curve runs out at 296. Past that, more than a quarter of every step is
spent agreeing rather than computing, and by the time all 2,048 are working, less than a third of
each step is arithmetic. The bill for agreeing stops growing. The work it is being compared against
does not stop shrinking.

Worth being clear about what the 296 is. It is not a law about clusters. It is what falls out of
this model, this buffer and this link speed, at a target somebody chose. Move any of those and it
moves.

### Some of the agreeing can happen while the arithmetic is still going

The two clocks above run one after the other, which is the pessimistic reading and the one this
notebook keeps. Real systems do better.

A step does not produce its whole buffer at the end. The last layer finishes first on the way back,
so a machine can start sending its share while still computing the rest, and anything sent during
time it was going to spend computing anyway costs nothing extra. That is what **hidden behind the
arithmetic** means, and the third slider asks what fraction of it you think you could arrange. At
zero you get the number this notebook reports everywhere else.

In [ ]:
#@title 🎛️ 3.4 — move the link, and how much of the agreeing hides { display-mode: "form" }
rv.scaling_lab()

### What those eight processes are actually running

Everything in this part has been arithmetic. The arrangement it prices has a name, **data
parallel**, and it is the eight processes you started in Part 1: one for each machine, each with its
own copy of the model and each about to be handed a different slice of the batch.

The three numbers Part 1 handed each process, `LOCAL_RANK`, `RANK` and `WORLD_SIZE`, are what turn
one script into eight cooperating ones:

```python
gpu     = int(os.environ["LOCAL_RANK"])     # which machine this process owns
model   = FDD71().to(gpu)                   # one copy of the model, on that machine
model   = DDP(model, device_ids=[gpu])      # the copies agree inside every backward
sampler = DistributedSampler(dataset)       # a different slice of the data for each

for x, y in loader:                         # then, every step
    x, y = x.to(gpu), y.to(gpu)             # the batch lands where the model is
```

**`DDP`** is where this part's arithmetic lives: inside every `backward` it averages the gradients
across all eight, which is the all-reduce we just priced. **`LOCAL_RANK`** is the line worth
remembering, because `RANK` in its place works perfectly on this box and breaks the day somebody
runs it on two.

In [ ]:
#@title 🧠 Quick check — doubling and getting 1.3 times { display-mode: "form" }
rv.quick_check("why_slower")

If one sentence survives this part: **adding machines does not automatically make training faster,
because the machines have to spend time agreeing with each other.** The slices and the hand-off
count are how you put a number on that, and a number is what turns an argument into a decision.

Every line of it assumed one machine can hold the model. That is the next thing to check.

---
# Part 4 · Does the model even fit?

Everything so far assumed a machine can hold the model while it works on it. That assumption has
been doing a lot of work, and one multiplication checks it.

In [ ]:
#@title 📊 4.1 — what one machine holds, for one parameter { display-mode: "form" }
rv.what_one_machine_holds()

### Five arrays, not one heavier parameter

It is worth being exact about what sixteen bytes a parameter means, because the natural reading of
it is wrong. **A parameter is still two bytes. Nothing was bolted onto it.** What training does is
keep four more arrays alongside the model, each one holding a number for every parameter, for as
long as the run lasts. Sixteen is what you get when you add those five arrays up and divide by the
number of parameters. It describes the run, not the parameter.

The distinction is worth the paragraph because it tells you what can be moved. A parameter has to
be wherever the arithmetic using it happens. A whole array does not, and section 5.3 is about
handing four of these five out across machines that already exist.

### Precision is a memory decision before it is anything else

Training moves each parameter by a very small amount each step. Stored narrowly, the smallest of
those moves are smaller than the format can represent, so they round to zero, and **an update below
that range is lost unless it is accumulated somewhere with more precision**. That is what the wider
copy is for: the model is used narrowly and improved widely.

The last two arrays belong to the **optimiser**, the procedure that decides how far each parameter
moves on a given step. It does better if it remembers something about the recent past rather than
looking only at the step in front of it, so it keeps two records: roughly how each parameter has
been moving, and how much that movement has been varying. Each record is one number per parameter,
which is why two of them come to twice the size of the model.

### The wall, in one division

Which copies the optimiser keeps is a choice somebody made rather than a property of the model, and
a procedure that keeps fewer of them moves the number below. Turning that into a yes or a no takes one division:

$$\text{largest model} = \frac{\text{memory on one machine}}{\text{bytes for every parameter}}$$

Here the memory on one machine is the 96 GB in the brief, the bytes for every parameter is what you
just counted, and the largest model is the biggest one a single machine could train at all. In
words: divide what the machine holds by what one parameter costs.

### 🎯 Task 5 · find the wall

> 💡 **Hint** · The first: how many parameters there are, times what each one costs, converted from
> bytes into GB. The second is that same relationship turned around, so divide where you multiplied:
> the memory you have, in bytes, over what one parameter costs.

In [ ]:

def model_state_gb(parameters, bytes_per_parameter):
    """What one machine must hold to take a step, if it holds the whole model."""
    # 🎯 TASK 5 — the memory bill, and the wall it puts you against
    #   parameters x bytes_per_parameter is a number of bytes, and 1e9 bytes is one GB
    return ???                          # every parameter, at that many bytes, in GB


def largest_model_that_fits(memory_gb, bytes_per_parameter):
    #   memory_gb x 1e9 is the bytes you have; one parameter costs bytes_per_parameter
    return ???                          # how many parameters that memory pays for


for _name, _n in [("a 1 billion model", 1e9), ("an 8 billion model", 8e9),
                  ("the model in the brief", the_brief.PARAMETERS)]:
    _gb = model_state_gb(_n, the_brief.BYTES_PER_PARAMETER)
    print(f"{_name:<24} {_gb:>8,.0f} GB   {'fits' if _gb <= the_brief.MEMORY_GB else 'does not fit'}")

_wall = largest_model_that_fits(the_brief.MEMORY_GB, the_brief.BYTES_PER_PARAMETER)
print(f"\nlargest model one machine could train alone, nothing shared: "
      f"{_wall/1e9:.1f} billion parameters")
_ok = (abs(model_state_gb(70e9, 16) - 1120) < 1e-6
       and abs(largest_model_that_fits(96, 16) - 6e9) < 1)
print("1,120 GB for the brief's model, and a wall at 6.0 billion ✓" if _ok
      else "✗ 70 billion at 16 bytes is 1,120 GB, and 96 GB pays for 6.0 billion "
           "parameters. Check where the conversion into GB went")

In [ ]:
#@title 📊 4.2 — what fits in one machine, and for which job { display-mode: "form" }
rv.what_fits()

### What the division assumed

That is one wall of several. The division priced the **plain arrangement**, the one the notebook
has assumed since Part 2, and three choices are baked into it:

| What the division assumed | What would move it, and where |
|---|---|
| the brief's optimiser and precision, at 16 bytes | a procedure that keeps fewer copies |
| one machine holding the whole thing by itself | splitting it across machines · Part 5 |
| every value kept, nothing recomputed | recomputing instead of storing · 5.3 |

Not one of the three is a law. Most of this notebook is about overturning them one at a time, and
what the division gives you is the wall standing in front of the plan as written.

### What this count leaves out

The count is optimistic inside that arrangement too. It counts the five arrays and nothing else,
leaving out:

- the values kept from the forward pass for the backward pass to use, which at a long context can
  outweigh everything above;
- a fixed overhead of a gigabyte or two on every machine before your model loads at all;
- the temporary buffers the agreeing needs while it is happening;
- and memory that is free but unusable, because it is in the wrong sized pieces.

So **this is a lower bound, and a configuration that only just fits does not fit.** Every time this
notebook says something fits, it means the model state fits and the rest needs measuring.

In [ ]:
#@title 🧠 Quick check — running a model against training one { display-mode: "form" }
rv.quick_check("inference_or_training")

It is worth naming what has just changed, because it is the hinge of the notebook.

Parts 2 and 3 **split the data**: every machine held its own copy of the whole model and worked on
a different slice of the batch. The weighting, the ring and the agreeing all describe that one
arrangement, and part 4 has just shown it cannot start, because the copy does not fit.

So the next part **splits the model**. The machines stop each holding a copy and start holding a
piece of one. There are three places to cut it, and each one costs something different.

---
# Part 5 · Three ways to cut a model that does not fit

A model is a stack of layers, each layer a large block of numbers, applied one after another. There
are exactly three things you can cut, large runs generally combine them, and each one buys memory
by selling something else back.

## 5.1 · Cut across a layer

Give each machine part of every layer's arithmetic. That buys memory, since no machine holds the
whole layer, and pays for it in agreeing: the moment one piece needs what another worked out, the
two have to exchange something. There are two ways to cut a layer and they differ in how much.

In [ ]:
#@title 📊 5.1 — two ways to cut one layer { display-mode: "form" }
rv.two_ways_to_slice()

### Two conditions, and they fail for different reasons

The second way is what a real layer needs, so that exchange happens over and over, all the way down
the stack. Before its cost is even relevant there are two conditions.

**The shape has to divide.** You cannot cut a layer more finely than it is built. The width has to
split evenly. So does the number of **attention heads**, which are the separate comparisons a layer
runs side by side, and so does the smaller number of those that carry keys and values. That last one
is usually the binding constraint and it is the one people forget.

### The second condition is about the cable, not the model

**The moving has to hide behind the arithmetic.** Moving data is only cheap while the machines are
busy with something else. If they finish their arithmetic and sit waiting for data, it is the link
that sets the pace rather than the chips you bought. So what matters is how much has to move,
against how fast the link carries it. Under one and the moving disappears into work that was
happening anyway. Over one and the machines are waiting.

That second test uses peak figures, so passing it means a cut is not obviously impossible. It does
not promise the overlap actually happens, which still needs measuring.

### 🎯 Task 6 · the two conditions on a cut

> 💡 **Hint** · The first: three separate remainders, and every one of them has to come out zero, so
> the answer is one thing being true and another and another. The second: how much has to move
> against how fast the link carries it, so it is a ratio, and both halves of it are named in the
> paragraphs above.

In [ ]:

def shape_divides(degree, hidden, heads, kv_heads):
    # 🎯 TASK 6 — the two conditions a cut across a layer has to pass
    #   hidden % degree, heads % degree and kv_heads % degree all have to be 0,
    #   and the answer is the three of them together
    return ???                          # all three have to split evenly


def hides_behind_the_arithmetic(degree, hidden, peak_flops, link_bytes_s):
    #   (degree - 1) slices out of (2 x hidden) have to move, and that fraction is
    #   charged against peak_flops / link_bytes_s
    return ???                          # how much must move, against how fast the link is


for _d in (4, 8, 16):
    _ok = shape_divides(_d, the_brief.HIDDEN, the_brief.HEADS, the_brief.KEY_VALUE_HEADS)
    print(f"degree {_d:>2}   the shape divides: {_ok}")
print()
for _d in (4, 8):
    for _where, _link in [("inside a box", the_brief.LINK_IN_A_BOX),
                          ("across the fabric", the_brief.LINK_ACROSS)]:
        _r = hides_behind_the_arithmetic(_d, the_brief.HIDDEN, the_brief.PEAK_FLOPS, _link)
        print(f"degree {_d} {_where:<18} ratio {_r:6.3f}   "
              f"{'the moving hides' if _r <= 1 else 'the moving does not hide'}")
_ok = (shape_divides(8, the_brief.HIDDEN, the_brief.HEADS, the_brief.KEY_VALUE_HEADS)
       and not shape_divides(16, the_brief.HIDDEN, the_brief.HEADS, the_brief.KEY_VALUE_HEADS)
       and abs(hides_behind_the_arithmetic(8, the_brief.HIDDEN, the_brief.PEAK_FLOPS,
                                           the_brief.LINK_IN_A_BOX) - 0.4747) < 1e-3)
print("\neight divides and sixteen does not, and eight hides at 0.475 in a box ✓" if _ok
      else "\n✗ degree 8 should divide and 16 should not, and degree 8 inside a box "
           "should come out at 0.475. All three remainders matter, and so does the 2")

### On these numbers, no degree of this cut survives leaving the box

Cutting sixteen ways fails before the link is even considered, because only eight of the heads carry
keys and values and eight does not divide sixteen.

The two that do divide tell the same story about placement. Inside a box the ratio is comfortably
under one. Across the fabric it is seven and seventeen, and the answer to how far this cut can
stretch between boxes is not a number, it is nowhere.

That is a boundary rather than a preference, and it is the only rule in this notebook that decides
something before anybody measures anything.

## 5.2 · Cut along the stack

Give each machine a run of consecutive layers instead. Work enters at the first machine, and each
one hands its results to the next.

The problem is obvious once it is drawn: while the work is still climbing the stack, most of the
machines have nothing to do.

In [ ]:
#@title 🎬 5.2 — the schedule filling in { display-mode: "form" }
rv.the_schedule()

### Feed it more micro-batches and the waiting shrinks, but never to nothing

A machine's share of the batch does not arrive all at once. It is fed through in pieces called
**micro-batches**, and more of them keep the machines busy: one can start the next as soon as it has
handed the last one on. What is left is the warm-up at the start and the drain at the end.

$$\text{share waiting} = \frac{\text{stages} - 1}{\text{micro-batches} + \text{stages} - 1}$$

A **stage** is one position in the stack, which throughout this notebook is one machine. In words: a
machine waits for the work to reach it and again for it to finish behind it, and everything else is
the run.

### 🎯 Task 7 · the share of a step spent waiting

> 💡 **Hint** · Count slots. A machine waits one slot for every stage above it on the way up and the
> same again on the way down, which comes to one fewer than the stages. The run is that many slots
> longer than the micro-batches. Waiting over the whole run is the answer.

In [ ]:

def idle_share(stages, micro_batches):
    """The share of the machine that is waiting rather than working."""
    # 🎯 TASK 7 — the share of a step a machine spends waiting
    return ???                # slots a machine waits, over slots in the run


print("            " + "".join(f"{_m:>8}" for _m in (1, 2, 4, 8, 16, 32, 64)))
for _s in (4, 8, 16):
    print(f"{_s:>3} stages " + "".join(f"{idle_share(_s, _m)*100:>7.1f}%"
                                       for _m in (1, 2, 4, 8, 16, 32, 64)))
_ok = abs(idle_share(4, 16) - 3 / 19) < 1e-9 and abs(idle_share(8, 1) - 7 / 8) < 1e-9
print("\nfour stages over sixteen micro-batches waits 15.8%, and over one waits 87.5% ✓"
      if _ok else "\n✗ four stages over sixteen micro-batches should be 3/19, which is "
                  "15.8%. Count the slots above and below, and the length of the run")

### More micro-batches is not a free dial

Read the table two ways. Along a row, more micro-batches always helps. Down a column, more stages
always hurts, because there is more stack to climb before the last machine sees anything. That is
not an argument for fewer machines. Stages are what you spent to make the model fit at all, and this
is the price of that, quoted separately.

There is a scheduling trick worth knowing about, because people describe it as fixing this and it
does not. Reordering the work so each machine alternates between going forward and coming back does
not change the formula above at all. What it changes is how much a machine has to hold at its
busiest moment, which lets you raise the number of micro-batches, and that is what shrinks the
waiting. The gain is real and it arrives indirectly.

### What caps the micro-batches is the batch itself

They have to add up to the batch the brief asked for, and that batch is a product of four things:

$$\text{batch} = \text{context} \times \text{sequences per micro-batch} \times
\text{micro-batches} \times \text{copies}$$

The context is how many tokens are in one sequence, and the copies are how many machines hold their
own version of the model. The brief fixes the batch at 8,388,608 tokens, so moving one of the four
means moving another: **you cannot raise the micro-batches without taking machines away from
somewhere else.**

A batch has a useful range, which is what makes that a constraint rather than a detail. Too small
and the machines spend their time coordinating. Too large and the extra tokens stop teaching the
model anything, so you are paying for data that no longer helps.

## 5.3 · Cut the stored copies

The third cut does not divide the arithmetic at all. Every machine does the same work on the same
data and gets the same answer. The only thing that changes is what it is holding while it does so.

Most of the sixteen bytes are not being read at any given instant. The wider copy and the
optimiser's two are consulted once a step, so instead of every machine keeping its own set, the
copies are shared out across the machines and fetched when needed. Three levels, each one sharing
out one more thing: first what the optimiser remembers, then the directions of travel, then the
model itself. You will hear them called ZeRO stages, or FSDP when the last one is included.

In [ ]:
#@title 📊 5.3 — what one machine holds at each level { display-mode: "form" }
rv.what_stays_resident()

### Written out level by level, because dividing the whole bill would be wrong

Each level shares out a different part of the bill and leaves the rest whole on every machine, so
there is no single number to divide. Writing it as what stays plus what is shared is the only way
to get it right, and it also makes clear how little is left by the end.

### 🎯 Task 8 · what stays on a machine at each level

> 💡 **Hint** · Sixteen bytes, moved one item at a time from the first line to the second as the
> level goes up: the optimiser's twelve, then the two narrow bytes of direction, then the two of the
> model itself. Four levels, so each line is four numbers written out by hand, and the two lines add
> to sixteen at every one of them.

In [ ]:

def resident_bytes_per_parameter(level, copies):
    """What one machine still holds per parameter, once `level` items are shared.

    level 0  nothing shared          level 2  also the directions of travel
    level 1  the optimiser's three   level 3  also the model itself
    """
    # 🎯 TASK 8 — what each sharing level leaves on a machine
    whole  = ???            # bytes that stay on every machine at this level
    shared = ???            # bytes that get divided across the copies
    return whole + shared / copies


for _lvl, _name in enumerate(["nothing shared", "the optimiser's copies", "also the directions",
                              "also the model"]):
    _b = resident_bytes_per_parameter(_lvl, 64)
    print(f"{_name:<24} {_b:>6.3f} bytes a parameter   "
          f"{_b * the_brief.PARAMETERS/32/1e9:>6.2f} GB a machine, with the model cut 32 ways")
_ok = all(abs(resident_bytes_per_parameter(_l, 64) - _v) < 1e-9
          for _l, _v in [(0, 16), (1, 4 + 12 / 64), (3, 0.25)])
print("\n16 bytes whole, 4.19 once the optimiser is shared, 0.25 at the end ✓" if _ok
      else "\n✗ level 0 should be 16, level 1 should be 4 + 12/64, and level 3 should be "
           "0.25. The two lines add to 16 at every level")

### This cut solves memory and nothing else, and it was never asked to

Sharing everything across all 2,048 machines brings the model state down to about half a gigabyte.
It fits with room to spare, and it is not a plan: with every machine holding a copy the batch cannot
be smaller than 16.8 million tokens against the 8.39 the brief asked for, and less than a third of
each step would be arithmetic. Part 6 will show that on the panel.

There is also something none of the three levels touches. The values kept from the forward pass are
different on every machine by construction, so there is nothing to share. The repair for those is to
keep only some of them and recompute the rest on the way back, which trades time for space:
Part 1's six operations a parameter a token become about eight, a third more arithmetic on a budget
you have already priced.

In [ ]:
#@title 🧠 Quick check — a cut across four boxes { display-mode: "form" }
rv.quick_check("across_boxes")

Three cuts, one cluster, and they all want the same machines.

| The cut | What it divides | What it costs |
|---|---|---|
| across a layer | the arithmetic inside every layer | a round of agreeing, in every layer, every step |
| along the stack | the layers, into consecutive runs | machines waiting at the start and the end |
| the stored copies | what is held, never what is done | more data moved, and nothing else |

What is left is deciding how to divide the machines up.

---
# Part 6 · Which configuration do we run?

Nothing new from here. A **gate** is one requirement a configuration has to satisfy, and there are
six of them. Every one on the panel below runs on a function you wrote:

| | The gate | Where it came from |
|---|---|---|
| 1 | the product is valid | the dials, which cannot make anything else |
| 2 | the cut is legal | the two conditions in 5.1 |
| 3 | the model state fits | the memory bill in Part 4, and the sharing levels in 5.3 |
| 4 | the batch contract holds | the batch equation in 5.2 |
| 5 | efficiency meets the target | the step time in Part 3 and the waiting in 5.2 |
| 6 | it meets the date | the same step time, multiplied out |

Two dials: how far the model is cut across a layer, and how far along the stack. Whatever is left of
the 2,048 machines becomes copies.

In [ ]:
#@title 🎛️ 6.1 — the builder { display-mode: "form" }
rv.builder(globals())

### Three verdicts, because feasible is not one thing

**Valid** means the arithmetic holds: the machines divide up, the cut is legal, and the training
problem is still the one that was approved. **Meets the brief** adds the three targets: memory,
efficiency and the date. **What you would recommend** is not a gate at all, and Part 7 is about the
distance between the last two.

Gate 3 never says "it fits". It says the model state fits, and a benchmark is still owed.

One more thing the panel shows that a simpler one would miss: **gate 5 can fail while every
hardware gate passes**, which means the run works, finishes on time, and is training a different
problem from the one that was signed off.

### The cut made the agreeing cheaper, and nothing before this said so

Part 3 priced a round of agreeing at 140 GB, because every machine held a whole copy of the model.
These configurations do not. Cutting the model thirty two ways cuts what the copies have to agree
over by the same thirty two, so the direction of travel they exchange is **4.4 GB rather than 140**,
and the step times on the panel are computed with that discount already applied.

It is worth pausing on, because everywhere else in this notebook buying memory has cost time. Part 5
priced what cutting across a layer costs: a round of agreeing inside every layer, every step. This
is the other side of that same cut, and it is a discount rather than a trade.

In [ ]:
#@title 📊 6.2 — three configurations, side by side { display-mode: "form" }
rv.three_configurations(globals())

### A and B: same machines, same memory, nearly three and a half days apart

Take the first two columns. Both cut the model thirty two ways, so both hold exactly 35 GB on every
machine, both pass the same first four gates, and both arrive before the date. A memory report cannot tell them apart. Memory
was never the thing that separated them.

One number separates them. Cutting across four and stacking over eight leaves each machine waiting
30.4 percent of the time. Cutting across eight and stacking over four leaves it waiting 15.8. Nothing
was bought and nothing was repaired: four machines moved from one kind of cut to another.

That shows up in gate 5 and nowhere else. It is also the gate easiest to wave through, because the
one that fails it still arrives four and a half days early and nobody was promised an efficiency
number in a meeting. Take the other one, and Part 7 is where you find out how much that was worth.

Worth trying on the panel before moving on: cut across sixteen. It reaches 88.8 percent and would
finish in fifteen days, which is better than either of these on every number a manager would ask
about. It is also not legal, because eight key and value heads do not divide sixteen ways. No amount
of speed fixes a shape.

### C: half a gigabyte of ninety six, and still not a plan

Now the third column. C cuts nothing and shares everything instead, so the model state falls to
about half a gigabyte on a machine that holds ninety six. **Memory has stopped being the
constraint**, which is worth saying plainly, because the obvious next thought is that all that free
space should be filled with a larger model. It should not. Not one of the three gates it fails has
anything to do with space.

With every machine holding a copy, the smallest batch it can run is twice the one the brief
approved, less than a third of each step is arithmetic, and it lands nineteen days after the date.
Solving the constraint that was in front of you does not make the others go away, and Part 4's
question was only ever the first of six.

You have a plan. It passes every gate and it lands eight days early. Before signing it, it is worth
asking what the plan does not know.

---
# Part 7 · What does the plan not know?

Everything so far assumed every machine runs at the same speed, that nothing stops, and that the
requirement does not change. None of those is true of a run that takes weeks.

Three questions, one line of arithmetic each. None of them changes the plan. They change how much
confidence it deserves.

In [ ]:
#@title 📊 7.1 — two changes, and whether the plan notices { display-mode: "form" }
rv.does_the_plan_notice(globals())

### The plan is right about one of those and blind to the other

**The slow machine.** A machine at eighty percent speed does twenty percent less work per second,
and the run takes twenty five percent longer, because everything is divided by that eighty. It is
charged on the arithmetic only, since the links are unaffected, and that is exactly why the two
configurations part with the date between them. Gate 5 was measuring how much of each step was
arithmetic, and a slow machine is a tax on precisely that, so the one with less of it to spare runs
out first. The gate that looked like a technicality in Part 6 was the early warning.

**The longer context.** Here everything in our model holds, including the step time, and that is
the finding rather than a reassurance. Our model charges a flat rate for every token, so the work
that grows with the length of the context is not in it anywhere: a plan built on this arithmetic
will report that doubling the context is free. It is not. What the model can tell you is what the
change takes from the copies. What it cannot tell you is what it costs on top, and that is the
honest thing to say in the meeting.

In [ ]:
#@title 📊 7.2 — the run stops { display-mode: "form" }
rv.the_run_stops()

### The cheapest interval, and the assumption underneath it

Writing a checkpoint costs time every interval. Stopping costs the work since the last one. Those
pull in opposite directions, and the cheapest interval is where they are equal:

$$\text{overhead} = \frac{\text{write}}{\text{interval}} +
\frac{\text{interval}}{2 \times \text{time between stops}}$$

Here the write is how long a checkpoint takes, the interval is how often you take one, and the time
between stops is how long the cluster typically runs before something fails. In words: pay a little
often, or risk losing a lot rarely.

At the brief's twelve hours that lands on an interval of 16.4 minutes and an overhead of 2.3
percent, which turns A's 17.1 days into 17.5.

The time between stops is not measured anywhere in this notebook. It is declared in the brief, and
it is exactly the kind of number a plan inherits from whoever wrote the template last. The panel on
the right is how much of the answer that one assumption is carrying: at three hours the cheapest
interval is eight minutes, and at two days it is over half an hour.

**Change one number in the brief and the recommendation moves.** A faster fabric, a larger machine,
a different optimiser: none of them is a property of the model, and all of them change the answer.
That is what makes this a configuration decision rather than a fact about the model.

---
# Six questions, and what is behind each

| The question | What it is | What it takes to answer |
|---|---|---|
| Does the arithmetic divide? | the cuts multiply to the machines you own | one multiplication |
| Is the cut legal? | the shape divides, and the moving hides behind the work | two conditions, and the second needs measuring |
| Does it fit? | the model state against one machine's memory | bytes a parameter, times parameters, divided by the sharing |
| Is it the same problem? | the batch is still the one that was approved | context, sequences, micro-batches, copies |
| Is it worth the machines? | the share of a step spent on arithmetic | the two clocks, and the waiting in the stack |
| Does it land? | steps times the step time | everything above |

The first four are arithmetic and give the same answer for everyone. The last two are arithmetic
measured against a target somebody chose, which is a different kind of question and the one worth
arguing about in a meeting.

**Where this goes next.** Search for how large runs overlap their agreeing with their arithmetic, for
what a scheduler does when a machine fails mid-run rather than between steps, and for how the useful
range of a batch is actually established. All three are places where the arithmetic here stops and
measurement starts.

In [ ]:
#@title 🧠 Final check — one of these is false { display-mode: "form" }
rv.quick_check("wrapup")